# Chapter 10: Natural Language Processing Basics
## Notebook 01 — NLP Fundamentals

This notebook introduces the core building blocks of NLP: **tokenization**, **text preprocessing**, **text representation** (Bag of Words, TF-IDF, embeddings), and your first **sentiment analysis** pipeline.

### What you'll learn

| Topic | Section |
|-------|--------|
| Text preprocessing (tokenization, stemming, lemmatization) | §2 |
| Bag of Words, TF-IDF, one-hot encoding | §3 |
| Reusable preprocessing pipeline | §4 |
| Word embeddings (GloVe, similarity, analogies) | §5 |
| First sentiment analysis with TF-IDF + logistic regression | §6 |

**Estimated time:** 2.5–3 hours

---
*Generated by Berta AI*

---
## 1. Introduction & Setup

We'll use **NLTK** for tokenization and stemming/lemmatization, **spaCy** (optional) for advanced NLP, **scikit-learn** for TF-IDF and classifiers, and **pandas** for data. Run the cell below to install NLTK data if needed.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4)
np.random.seed(42)

print("Setup complete.")

---
## 2. Text Preprocessing Fundamentals

### Tokenization

**Word-level** tokenization splits text into words; **sentence-level** into sentences. NLTK's `word_tokenize` and `sent_tokenize` are standard choices.

In [ ]:
from nltk import word_tokenize, sent_tokenize

sample = "Natural language processing is amazing. It helps machines understand text!"
words = word_tokenize(sample)
sents = sent_tokenize(sample)
print("Words:", words)
print("Sentences:", sents)

### Stemming vs. Lemmatization

- **Stemming** chops off affixes (e.g. "running" → "run") — fast but can be crude.
- **Lemmatization** maps words to dictionary base form (e.g. "better" → "good") — needs POS for accuracy.

In [ ]:
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

test_words = ["running", "runs", "easily", "better", "studies"]
for w in test_words:
    print(f"{w:12} -> stem: {stemmer.stem(w):10}  lemma: {lemmatizer.lemmatize(w)}")

### Stop word removal

Stop words ("the", "is", "at") often add noise for many tasks. Remove them when building BoW/TF-IDF for classification.

In [ ]:
from nltk.corpus import stopwords

sw = set(stopwords.words('english'))
tokens = word_tokenize(sample.lower())
filtered = [t for t in tokens if t not in sw and t.isalnum()]
print("With stopwords:", tokens)
print("Without stopwords:", filtered)

---
## 3. Text Representation (Part 1)

### Bag of Words (BoW)

Each document is a vector of word counts. Order is lost but frequency is kept.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are pets"
]
cv = CountVectorizer()
X_bow = cv.fit_transform(docs)
print("Vocabulary:", cv.get_feature_names_out())
print("BoW matrix (dense):")
print(pd.DataFrame(X_bow.toarray(), columns=cv.get_feature_names_out()))

### TF-IDF

**Term Frequency–Inverse Document Frequency** downweights terms that appear in many documents (e.g. "the") and highlights discriminative terms.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(docs)
print("TF-IDF matrix (dense):")
print(pd.DataFrame(np.round(X_tfidf.toarray(), 3), columns=tfidf.get_feature_names_out()))

---
## 4. Practical Project: Text Preprocessing Pipeline

Using the chapter's `text_preprocessing` module we build a reusable cleaner and show before/after.

In [ ]:
from text_preprocessing import clean_text, TextPreprocessor

raw = "I've been learning NLP! It's really COOL. We're building pipelines."
cleaned = clean_text(raw, remove_stop=True, lemmatize=True)
print("Before:", raw)
print("After:", cleaned)

preprocessor = TextPreprocessor(remove_stopwords=True, lemmatize=True)
texts = ["Cats are running.", "Dogs run fast.", "The cat runs."]
preprocessor.fit(texts)
transformed = preprocessor.transform(texts)
print("\nTransformed:", transformed)

---
## 5. Introduction to Word Embeddings

**Word embeddings** map words to dense vectors so that similar words are close in space. **GloVe** and **Word2Vec** are classic methods. Below we load GloVe (if you have a file) or use a small in-notebook example.

In [ ]:
# If you have GloVe file (e.g. glove.6B.100d.txt), set path. Otherwise we use a tiny demo.
GLOVE_PATH = None  # e.g. "../data/glove.6B.100d.txt"

if GLOVE_PATH and os.path.exists(GLOVE_PATH):
    from embedding_utils import load_pretrained_embeddings, find_similar_words, word_analogy
    embeddings, _ = load_pretrained_embeddings("glove", path=GLOVE_PATH, dim=100)
    print(f"Loaded {len(embeddings)} words.")
    sim = find_similar_words("king", embeddings, n=5)
    print("Similar to 'king':", sim)
    analogies = word_analogy("king", "man", "woman", embeddings, n=5)
    print("king - man + woman =", analogies)
else:
    # Demo: random embeddings for a small vocab (for illustration only)
    vocab = ["king", "queen", "man", "woman", "paris", "france", "good", "great"]
    np.random.seed(42)
    emb = {w: np.random.randn(20).astype(np.float32) for w in vocab}
    from embedding_utils import cosine_similarity, find_similar_words
    # Make queen ≈ king and woman ≈ man for demo
    emb["queen"] = emb["king"] + 0.3 * np.random.randn(20).astype(np.float32)
    emb["woman"] = emb["man"] + 0.3 * np.random.randn(20).astype(np.float32)
    sim = find_similar_words("king", emb, n=4)
    print("Similar to 'king' (demo):", sim)

---
## 6. First NLP Task: Sentiment Analysis

We load the chapter's **reviews** dataset, preprocess, build TF-IDF features, and train a **logistic regression** classifier. Then we evaluate and test on new examples.

In [ ]:
DATA_DIR = os.path.join('..', 'datasets')
reviews_path = os.path.join(DATA_DIR, 'reviews.csv')

if os.path.exists(reviews_path):
    df = pd.read_csv(reviews_path)
    print(df.head())
    print("\nSentiment distribution:", df['sentiment'].value_counts())
else:
    # Fallback: small synthetic data
    df = pd.DataFrame({
        'text': ["Great movie, loved it!", "Terrible film.", "Amazing product.", "Waste of money."],
        'sentiment': [1, 0, 1, 0],
        'rating': [5, 1, 5, 1]
    })
    print("Using fallback data:")
    print(df)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df['text'].fillna('').values
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf = tfidf.transform(X_test)

clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(X_train_tf, y_train)
y_pred = clf.predict(X_test_tf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Test on new examples
new_texts = ["This product is fantastic!", "Really disappointed."]
X_new = tfidf.transform(new_texts)
preds = clf.predict(X_new)
proba = clf.predict_proba(X_new)
for t, p, pr in zip(new_texts, preds, proba):
    print(f"'{t}' -> {'Positive' if p == 1 else 'Negative'} (conf: {pr[p]:.2f})")

---
## 7. Key Takeaways

- **Tokenization** (word/sentence) is the first step; then **cleaning** (lowercase, stopwords, lemmatization) and **vectorization** (BoW, TF-IDF or embeddings).
- **TF-IDF** is simple and effective for classification; **word embeddings** capture similarity and analogies.
- **Sentiment analysis** can be done with TF-IDF + logistic regression as a strong baseline.

Next: **Notebook 02** — deep learning for text (CNNs, LSTMs), multi-class classification, and NER.

---
*Generated by Berta AI*